# EUV2MAG baseline

adapted from the baseline template by Andrés Muñoz-Jaramillo

This notebook trains a deliberately simple baseline for EUV2MAG: a per-pixel linear map from
the three EUV channels to the magnetogram. It assumes you have been through
`0_euv2mag_dataset_dataloader.ipynb`.

The point of the baseline is not to be good. It is to answer the question the fine-tune
cannot: **how much of the magnetogram is predictable from EUV intensity at the same pixel,
with no spatial context at all.** Whatever Surya gains over this number in
`2_euv2mag_finetune.ipynb` is what the backbone's spatial and spectral structure is buying.

## Set your cuda visible device

Since we are sharing resources, please make sure you are using the GPU you were assigned.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger

# Append base path.  May need to be modified if the folder structure changes.
sys.path.append("../../")

from workshop_infrastructure.utils import build_scalers

torch.set_float32_matmul_precision("medium")

## Load configuration

The same typed config the training script reads, so the same YAML behaves identically here
and in production.

In [ ]:
from downstream_apps.Imagetranslation.configs import load_euv2mag_config

cfg = load_euv2mag_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")
print(f"  {'+'.join(cfg.data.input_channels)} -> {'+'.join(cfg.data.target_channels)}")

## Download assets

The linear baseline needs no backbone, so we skip the 1.8 GB of weights and fetch only the
scalers.

In [ ]:
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers"])

scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")

## Define the datasets and dataloaders

Unlike notebook 0, we do not spell the dataset arguments out by hand here.

In [ ]:
from downstream_apps.Imagetranslation.datasets.euv2mag_dataset import Euv2MagDataset

In [ ]:
# build_helio_dataloaders() constructs the train and validation datasets and wraps them
# in DataLoaders. It fills in every generic argument (channels, temporal sampling,
# resolution, S3 access, worker settings) from the config — see notebook 0 for what that
# block looks like written out. Only the EUV2MAG-specific arguments are passed here, which
# is exactly the list you replace when you fork this app.
#
# It also handles three details that are easy to get wrong by hand:
#   * the validation set gets phase="val" (no random channel masking or flips),
#   * only the training loader shuffles, and only it drops a ragged last batch, and
#   * train and validation are capped SEPARATELY, from data.max_train_samples and
#     data.max_val_samples — so you can vary the amount of training data and still score
#     every run on the same validation set.
from workshop_infrastructure.datasets.builders import build_helio_dataloaders

train_data_loader, val_data_loader = build_helio_dataloaders(
    cfg,
    Euv2MagDataset,
    scalers=scalers,
    num_workers=4,          # fewer workers than the script: notebooks start faster
    seed=cfg.seed,
    #### Downstream (DS) specific parameters
    input_channels=cfg.data.input_channels,
    target_channels=cfg.data.target_channels,
)

batch_size = cfg.batch_size
print(f"train: {len(train_data_loader.dataset)} samples | "
      f"val: {len(val_data_loader.dataset)} samples | batch_size: {batch_size}")

# TIP: while you are iterating, set cfg.data.max_train_samples = 8 above. Every sample is
# a ~1 GB download the first time it is seen.

Training and validation get separate datasets and dataloaders. They differ only in the index they read and in `phase`: `phase="val"` turns off the random channel masking and vertical flips used for training augmentation.

The loaders use `multiprocessing_context="spawn"` — the dataset holds an S3 client that does not survive `fork`, and spawn also avoids lockups in shared environments.

In [ ]:
# Inspect a single batch to confirm shapes before training.
batch = next(iter(train_data_loader))
print({k: (tuple(v.shape) if hasattr(v, "shape") else type(v).__name__) for k, v in batch.items()})

## Define the simple baseline model

Defining a simple baseline is important to understand what value the AI model is bringing to
the problem.

It is always very good to have a very simple baseline model. Ideally one that cannot overfit
the data. This is a very good way of really measuring the value added of complex models.
Classical machine learning excels here:

- Regressions and logistic regressions.
- Climatological averages.
- Persistence.
- Simple transformations.

Simple models avoid excessively optimistic assessments of the capabilities of complex models,
and for many problems are actually remarkably hard to beat.

Here the simple model is a **1x1 convolution**: a learned linear combination of the input
channels at each pixel independently, with no spatial context whatsoever. Time is folded into
the convolution's input channels, so it can use several frames even though it has no notion
of their order.

In [ ]:
from downstream_apps.Imagetranslation.models.simple_baseline import Conv2DImageTranslationModel

model = Conv2DImageTranslationModel(
    input_channels=cfg.data.input_channels,
    target_channels=cfg.data.target_channels,
    n_input_timestamps=cfg.model.time_embedding.time_dim,
)
print(model)

Now we pass a batch through the model. Since it has not been trained and was initialized
randomly, the output has no meaning — this only tests that the forward pass has no dimension
problems. Dimension problems are the dominant source of error in this kind of work.

Note the model takes the **whole batch dictionary**, not a bare tensor. That is deliberate: it
makes the baseline interchangeable with the Surya model in notebook 2, so both share the same
Lightning module instead of needing a special wrapper to unwrap the dict.

In [ ]:
output = model(batch)
print("output  ", tuple(output.shape))
print("target  ", tuple(batch["forecast"].shape), "-> squeezed to",
      tuple(batch["forecast"].squeeze(2).shape))

## Define your metrics

Metrics are a very important part of training AI models. They provide the quantification of
error, which in turn shifts the weights towards better performing models. They also provide a
way for you to monitor performance, identify overfitting, and quantify value added.

`Euv2MagMetrics` controls which metrics act as the "loss" (the ones that backpropagate) and
which are only monitored. Four modes:

- `train_loss` — backpropagated (MSE).
- `val_loss` — logged as `val_loss`, and therefore **what selects the best checkpoint**.
- `train_metrics` / `val_metrics` — reported only. These do *not* select checkpoints.

**Both MSE and MAE are reported, and the pair matters here.** A magnetogram is dominated by
quiet Sun near zero with a thin tail of strong active-region field. MSE is driven almost
entirely by that tail; MAE tells you whether the quiet Sun is right. Watching only one of
them hides half the behaviour.

All of these are in **normalized** space, because that is what the model outputs. An MSE of
0.1 is 0.1 of a per-channel standard deviation, not 0.1 Gauss. For physical units, inverse
transform first — which is what the figure at the end of this notebook does.

In [ ]:
from downstream_apps.Imagetranslation.metrics.euv2mag_metrics import Euv2MagMetrics

train_loss_metrics = Euv2MagMetrics("train_loss")
# val_loss is the quantity logged as "val_loss" and used to pick the best checkpoint.
# It defaults to the same MSE as train_loss — override Euv2MagMetrics.val_loss to change it.
val_loss_metrics = Euv2MagMetrics("val_loss")
train_evaluation_metrics = Euv2MagMetrics("train_metrics")
# Reported only: val_metrics do NOT influence checkpoint selection.
validation_evaluation_metrics = Euv2MagMetrics("val_metrics")

Now they can be evaluated on the model's output and the ground truth. The target needs its
lead-time axis squeezed out first — `euv2mag_target()` is the single definition of that, and
the Lightning module applies it for you during training.

In [ ]:
from downstream_apps.Imagetranslation.lightning_modules.pl_euv2mag import euv2mag_target

target = euv2mag_target(batch)
assert output.shape == target.shape, (output.shape, target.shape)

print("train_loss  ", train_loss_metrics(output, target))
print("val_metrics ", validation_evaluation_metrics(output, target))

## Define your PyTorch Lightning module

In this workshop we use PyTorch Lightning to train our models. Lightning reduces the amount
of code required to implement a training loop compared with plain PyTorch (at the expense of
control and versatility).

The training mechanics — the batch contract, the loss and metric dicts, the logging — are the
same for every downstream task, so they live in
`workshop_infrastructure/lightning_modules/pl_base.py` and are imported rather than copied.
`Euv2MagLightningModule` is three lines on top of that base: all it supplies is `target_fn`,
which says how to get the target tensor out of the batch dict.

In [ ]:
from downstream_apps.Imagetranslation.lightning_modules.pl_euv2mag import Euv2MagLightningModule

## Set your global seeds

Since training AI models generally uses stochastic gradient descent, it is a good idea to fix
your random seeds so that your training exercise is reproducible.

In [ ]:
# cfg.seed, not a literal: the seed belongs to the experiment, and the script reads the
# same value.
L.seed_everything(cfg.seed, workers=True)

## Initialize the Lightning module

Now we initialize the Lightning module for training, passing the dictionary of metrics.

Plain Adam over everything is right here: there is no pretrained backbone, so there is
nothing that wants a different learning rate from anything else. Notebook 2 uses a different
class for exactly that reason.

In [ ]:
metrics = {
    "train_loss": train_loss_metrics,
    "val_loss": val_loss_metrics,
    "train_metrics": train_evaluation_metrics,
    "val_metrics": validation_evaluation_metrics,
}

lit_model = Euv2MagLightningModule(
    model, metrics, lr=cfg.learning_rate, batch_size=batch_size
)

## Logging

In order to properly compare experiments against each other, it is very useful to log
evaluation metrics where they can be compared against other training runs. In this workshop
we use Weights and Biases (WandB).

The first time you run WandB on a machine it will ask you to log in. To do so:

- Select option 2 (existing account). In VSCode the dialog opens a box at the top of your screen.
- Click on get API Key (this will open a browser).
- Generate API Key.
- Paste it in the dialog box at the top of your VSCode

In [ ]:
project_name = cfg.wandb_project
run_name = "baseline_experiment_1"  # give your run a descriptive name

wandb_logger = WandbLogger(
    entity=cfg.wandb_entity,  # set wandb_entity in the config; null = personal account
    project=project_name,
    name=run_name,
    log_model=False,
    save_dir="./wandb/wandb_tmp",
)

csv_logger = CSVLogger("runs", name=project_name)

## Initialize the trainer

The trainer defines several properties of your training run: the max number of epochs, where
it takes place, the loggers, the callbacks (here we save the model with the lowest validation
loss), and the logging frequency.

In [ ]:
# Everything here comes from the config, so this Trainer and the one in
# 3_finetune_euv2mag.py --train_baseline are the same Trainer. Override a value by
# editing cfg above (e.g. cfg.max_epochs = 2) rather than by hardcoding it here.
trainer = L.Trainer(
    max_epochs=cfg.max_epochs,
    accelerator="auto",
    devices="auto",
    precision=cfg.precision if torch.cuda.is_available() else "32-true",
    accumulate_grad_batches=cfg.accumulate_grad_batches,
    # Pinned the same way the script pins them, so a notebook result and a script result
    # are comparable. See training.deterministic in the YAML.
    deterministic=cfg.deterministic,
    benchmark=False,
    logger=[wandb_logger, csv_logger],
    callbacks=[
        ModelCheckpoint(
            monitor="val_loss",
            mode="min",
            save_top_k=1,
        )
    ],
    log_every_n_steps=2,
)

## Fit the model

Finally we fit the model, passing the Lightning module and our dataloaders.

In [ ]:
trainer.fit(lit_model, train_data_loader, val_data_loader)

## Look at a prediction

The loss is a number; a magnetogram is a picture, and the two fail in different ways. A model
can have a respectable MSE and still predict a featureless grey disk, which is exactly what a
model that has learned only the mean looks like.

To read the picture we have to get back to **physical units**. The forward pipeline is
signum-log *then* z-score, so `inverse_transform_subset()` is what undoes both and returns
Gauss — and it handles the channel-ordering trap described in notebook 0.

In [ ]:
dataset = val_data_loader.dataset
lit_model.eval()

eval_batch = next(iter(val_data_loader))
device = next(lit_model.parameters()).device
on_device = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in eval_batch.items()}

with torch.no_grad():
    prediction = lit_model(on_device)

pred = prediction[0].float().cpu().numpy()
truth = eval_batch["forecast"][0].squeeze(1).float().cpu().numpy()

pred_phys = dataset.inverse_transform_subset(pred, dataset.target_channels)
truth_phys = dataset.inverse_transform_subset(truth, dataset.target_channels)

name = dataset.target_channels[0]
t, p = truth_phys[0], pred_phys[0]
# Symmetric limits taken from the truth and shared by both panels: a magnetogram is signed,
# and independently scaled panels would make a flat prediction look structured.
lim = float(np.percentile(np.abs(t), 99.5)) or 1.0

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (img, title, kw) in zip(axes, [
    (t, f"{name} truth", dict(cmap="gray", vmin=-lim, vmax=lim)),
    (p, f"{name} baseline prediction", dict(cmap="gray", vmin=-lim, vmax=lim)),
    (p - t, f"{name} residual (pred - truth)", dict(cmap="coolwarm", vmin=-lim, vmax=lim)),
]):
    im = ax.imshow(img, origin="lower", **kw)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, label="Gauss")
fig.tight_layout()

## Conclusion

We have now integrated the dataset, dataloaders, metrics and a baseline into an end-to-end
training loop, and we have a number to beat.

Keep the baseline's `val_loss` — the next step, `2_euv2mag_finetune.ipynb`, substitutes Surya
for the simple model and the comparison between the two is the whole point. A fine-tune that
does not beat a 1x1 convolution is not working, however plausible its loss curve looks.